# Credit Events and Restructuring

**Start here:** This deep dive expands on `02_pricing/pricing_across_asset_classes.ipynb`; use `02_pricing/pricing_fundamentals.ipynb` for the shared instrument JSON -> market -> model pipeline.

**Purpose:** Show three common distressed-credit workflows driven entirely by the Rust credit kernel.

**Prerequisites:** Basic familiarity with capital-structure priority, exchange offers, and liability management exercises.

**In this notebook:** We run the canonical recovery waterfall, compare hold-versus-tender economics for an exchange offer, and quantify an open-market repurchase LME.

> These are decision-support calculations, not mark-to-market valuations. Recovery allocation, exchange-offer economics, LME discount capture and all of their input validation live in `finstack_quant.core.credit`; the notebook only supplies inputs and formats the results.


## Concept

These analyses answer three common distressed-credit questions:

1. **Recovery waterfall** — who recovers what under a stressed distributable value, following the Absolute Priority Rule?
2. **Exchange offer** — is a hold-vs-tender trade-off economically attractive to holders?
3. **LME** — how much debt reduction and discount capture does a discounted liability-management move create?

In [1]:
## Presentation helpers

import sys
sys.path.insert(0, "../..")

from _shared import banner

def fmt_money(value: float) -> str:
    return f"${value:,.0f}"

def fmt_pct(value: float) -> str:
    return f"{value * 100:.2f}%"


## Recovery waterfall

The executable setup below imports the Rust-backed recovery API and maps the notebook's seniority labels to explicit numeric priorities. Lower numeric priorities recover first; equal-priority deficiencies share an insufficient residual estate pro rata.

The distributable estate includes pledged collateral. Haircut net collateral is allocated first and deducted from the estate before general recovery, preventing double counting.

In [2]:
## Exchange offer and LME analytics

# Both live in the Rust kernel alongside `allocate_recovery`, so input
# validation, canonical type names and the 2% tender hurdle are owned by
# `core::credit::liability_management` rather than by this notebook.
#
#   analyze_exchange_offer(old_pv, new_pv, consent_fee,
#                          equity_sweetener_value, exchange_type)
#       -> ExchangeOfferAnalysis
#   analyze_lme(lme_type, notional, repurchase_price_pct,
#               opt_acceptance_pct, ebitda)
#       -> LmeAnalysis
#
# Both return typed results (attribute access), not dicts.
from finstack_quant.core.credit import liability_management as lm

## Takeaways

- **`core.credit.recovery_waterfall.allocate_recovery()`** applies collateral and distributes the remaining estate by absolute priority.
- **`core.credit.liability_management.analyze_exchange_offer()`** reframes a restructuring proposal in holder economics rather than issuer rhetoric, and flags `tender_recommended` against a 2% hurdle over the hold value.
- **`core.credit.liability_management.analyze_lme()`** measures debt reduction, discount capture and leverage change from discounted liability-management moves.

All three are tested Rust kernels returning typed results, so the same numbers are reachable from Rust, Python and WASM. This notebook contributes only the inputs and the formatting.

In [3]:
import json
from pathlib import Path

from finstack_quant.core.credit.recovery_waterfall import (
    RecoveryClaim,
    allocate_recovery,
)

_NOTEBOOK_DATA = json.loads(Path("data/credit_events_and_restructuring.json").read_text())
SENIORITY_PRIORITY = {seniority: priority for priority, seniority in enumerate(_NOTEBOOK_DATA["SENIORITY_ORDER"])}

claims = [
    RecoveryClaim(
        id=claim.get("id", f"claim-{index}"),
        seniority=claim["seniority"],
        priority=SENIORITY_PRIORITY[claim["seniority"]],
        principal=claim["principal"],
        accrued=claim.get("accrued", 0.0),
        penalties=claim.get("penalties", 0.0),
        collateral=(claim["collateral_value"], claim.get("haircut", 0.0)) if "collateral_value" in claim else None,
    )
    for index, claim in enumerate(_NOTEBOOK_DATA["claims"])
]

In [4]:
## Example Run

banner("Recovery waterfall")
waterfall = allocate_recovery(100_000_000.0, claims)
print(f"Total distributed: {fmt_money(waterfall.total_distributed)}")
print(f"Residual         : {fmt_money(waterfall.undistributed_estate)}")
for row in waterfall.allocations:
    print(
        f"  {row.id:<20} claim={fmt_money(row.total_claim)} "
        f"recovery={fmt_money(row.total_recovery)} rate={fmt_pct(row.recovery_rate)}"
    )

banner("Exchange offer")
exchange = lm.analyze_exchange_offer(
    old_pv=45.0,
    new_pv=80.0,
    consent_fee=2.0,
    equity_sweetener_value=0.0,
    exchange_type="discount",
)
print(f"Tender total     : ${exchange.tender_total:.2f}")
print(f"Delta NPV        : ${exchange.delta_npv:+.2f}")
print(f"Breakeven recov. : {fmt_pct(exchange.breakeven_recovery)}")
print(f"Tender?          : {exchange.tender_recommended}")

banner("Open-market repurchase LME")
lme = lm.analyze_lme(
    lme_type="open_market",
    notional=200_000_000.0,
    repurchase_price_pct=0.60,
    opt_acceptance_pct=0.40,
    ebitda=25_000_000.0,
)
print(f"Cash cost        : {fmt_money(lme.cost)}")
print(f"Notional retired : {fmt_money(lme.notional_reduction)}")
print(f"Discount capture : {fmt_money(lme.discount_capture)}")
print(f"Holder impact    : {fmt_pct(lme.remaining_holder_impact_pct)}")
if lme.leverage_impact is not None:
    leverage = lme.leverage_impact
    print(f"Pre leverage     : {leverage.pre_leverage:.2f}x")
    print(f"Post leverage    : {leverage.post_leverage:.2f}x")

# Summary for test runner
{
    "waterfall_apr_satisfied": waterfall.apr_satisfied,
    "exchange_delta_npv": round(exchange.delta_npv, 2),
    "lme_discount_capture": round(lme.discount_capture, 2),
}


Recovery waterfall
Total distributed: $100,000,000
Residual         : $0
  first_lien_tl        claim=$51,000,000 recovery=$51,000,000 rate=100.00%
  sr_unsec_notes       claim=$82,000,000 recovery=$49,000,000 rate=59.76%
  sub_notes            claim=$41,000,000 recovery=$0 rate=0.00%
  common_equity        claim=$0 recovery=$0 rate=0.00%

Exchange offer
Tender total     : $82.00
Delta NPV        : $+37.00
Breakeven recov. : 100.00%
Tender?          : True

Open-market repurchase LME
Cash cost        : $48,000,000
Notional retired : $80,000,000
Discount capture : $32,000,000
Holder impact    : 0.00%
Pre leverage     : 8.00x
Post leverage    : 4.80x


{'waterfall_apr_satisfied': True,
 'exchange_delta_npv': 37.0,
 'lme_discount_capture': 32000000.0}